# Patch dataset preparation

This notebook creates patch index CSV files for 16x16, 32x32, and 64x64 raster patches centered on the final labeled samples. It does not train models and does not save patch arrays.

## 1. Load paths and parameters

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.patch_dataset import (
    DEFAULT_NODATA_VALUE,
    FACTOR_NAMES,
    RasterPatchDataset,
    audit_raster_alignment,
    create_patch_index,
    list_raster_files,
    save_patch_index,
)

raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
samples_csv = PROJECT_ROOT / "data/processed/samples/final_cluster_balanced_dataset.csv"
patch_dir = PROJECT_ROOT / "data/processed/patches"
patch_sizes = [16, 32, 64]
max_nodata_ratio = 0.0

patch_index_paths = {
    16: patch_dir / "labeled_patch_index_ps16.csv",
    32: patch_dir / "labeled_patch_index_ps32.csv",
    64: patch_dir / "labeled_patch_index_ps64.csv",
}

print(f"Raster folder: {raster_dir}")
print(f"Final labeled samples: {samples_csv}")
print(f"Patch index folder: {patch_dir}")
print(f"Patch sizes: {patch_sizes}")
print(f"max_nodata_ratio: {max_nodata_ratio}")

## 2. Load cleaned rasters and audit alignment

In [ ]:
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=DEFAULT_NODATA_VALUE)

print(f"Number of rasters: {len(raster_files)}")
for factor_name, path in zip(FACTOR_NAMES, raster_files):
    print(f"{factor_name}: {path.name}")
print(f"CRS: {raster_audit.crs}")
print(f"Raster size: {raster_audit.width} x {raster_audit.height}")
print(f"Resolution: {raster_audit.resolution}")
print(f"NoData: {raster_audit.nodata}")

## 3. Load final cluster-balanced labeled dataset

In [ ]:
samples_df = pd.read_csv(samples_csv)
required_columns = ["sample_id", "x", "y", "label", "source", "cluster_id", *FACTOR_NAMES]
missing = [column for column in required_columns if column not in samples_df.columns]
if missing:
    raise ValueError(f"Final labeled dataset is missing columns: {missing}")
samples_df = samples_df[required_columns].copy()

print(f"Total samples: {len(samples_df)}")
print("Label counts:")
print(samples_df["label"].value_counts().sort_index().to_string())
print("Cluster x label counts:")
print(pd.crosstab(samples_df["cluster_id"], samples_df["label"]).to_string())

## 4. Convert sample coordinates to raster row/col

In [ ]:
preview_index = create_patch_index(
    samples_df.head(5),
    raster_files,
    patch_size=16,
    max_nodata_ratio=max_nodata_ratio,
    factor_names=FACTOR_NAMES,
    nodata_value=DEFAULT_NODATA_VALUE,
)
print(preview_index[["sample_id", "x", "y", "row", "col"]].to_string(index=False))

## 5. Generate patch index for 16x16

In [ ]:
patch_index_16 = create_patch_index(samples_df, raster_files, 16, max_nodata_ratio, FACTOR_NAMES, DEFAULT_NODATA_VALUE)
saved_16 = save_patch_index(patch_index_16, patch_index_paths[16])
print(f"Saved: {saved_16}")
print(f"Valid patches: {int(patch_index_16['valid_patch'].sum())} / {len(patch_index_16)}")

## 6. Generate patch index for 32x32

In [ ]:
patch_index_32 = create_patch_index(samples_df, raster_files, 32, max_nodata_ratio, FACTOR_NAMES, DEFAULT_NODATA_VALUE)
saved_32 = save_patch_index(patch_index_32, patch_index_paths[32])
print(f"Saved: {saved_32}")
print(f"Valid patches: {int(patch_index_32['valid_patch'].sum())} / {len(patch_index_32)}")

## 7. Generate patch index for 64x64

In [ ]:
patch_index_64 = create_patch_index(samples_df, raster_files, 64, max_nodata_ratio, FACTOR_NAMES, DEFAULT_NODATA_VALUE)
saved_64 = save_patch_index(patch_index_64, patch_index_paths[64])
print(f"Saved: {saved_64}")
print(f"Valid patches: {int(patch_index_64['valid_patch'].sum())} / {len(patch_index_64)}")

## 8. Validate class and cluster balance after patch validity filtering

In [ ]:
patch_indices = {16: patch_index_16, 32: patch_index_32, 64: patch_index_64}
for patch_size, patch_index in patch_indices.items():
    valid = patch_index.loc[patch_index["valid_patch"]]
    invalid_count = len(patch_index) - len(valid)
    print(f"Patch size {patch_size}")
    print(f"total samples: {len(patch_index)}")
    print(f"valid patches: {len(valid)}")
    print(f"invalid patches: {invalid_count}")
    print("valid patches by label:")
    print(valid["label"].value_counts().sort_index().to_string())
    print("valid patches by cluster_id and label:")
    table = pd.crosstab(valid["cluster_id"], valid["label"])
    print(table.to_string())
    full_table = pd.crosstab(patch_index["cluster_id"], patch_index["label"])
    losses = full_table.subtract(table, fill_value=0)
    if (losses > 0).any().any():
        print("WARNING: Some samples became invalid because of edge or nodata filtering.")
        print(losses.astype(int).to_string())
    print("")

## 9. Define and test PyTorch lazy-loading Dataset

In [ ]:
smoke_test_passed = False
try:
    smoke_results = {}
    for patch_size in patch_sizes:
        dataset = RasterPatchDataset(
            patch_index_csv=patch_index_paths[patch_size],
            raster_dir=raster_dir,
            patch_size=patch_size,
            return_metadata=True,
            valid_only=True,
        )
        X, y, metadata = dataset[0]
        smoke_results[patch_size] = {
            "length": len(dataset),
            "X_shape": tuple(X.shape),
            "X_dtype": str(X.dtype),
            "y": int(y.item()),
            "metadata": metadata,
        }
        print(f"Patch size {patch_size}: length={len(dataset)}")
        print(f"X.shape: {tuple(X.shape)}")
        print(f"X.dtype: {X.dtype}")
        print(f"y: {int(y.item())}")
        print(f"metadata: {metadata}")
    smoke_test_passed = True
except ModuleNotFoundError as exc:
    print(f"RasterPatchDataset smoke test could not run: {exc}")

## 10. Print final summary

In [ ]:
print("Patch index paths:")
for patch_size in patch_sizes:
    print(f"{patch_size}: {patch_index_paths[patch_size]}")
print(f"number of total samples: {len(samples_df)}")
for patch_size, patch_index in patch_indices.items():
    valid = patch_index.loc[patch_index["valid_patch"]]
    print(f"Patch size {patch_size}: valid patches = {len(valid)}")
    print("valid label counts:")
    print(valid["label"].value_counts().sort_index().to_string())
    print("valid cluster x label table:")
    print(pd.crosstab(valid["cluster_id"], valid["label"]).to_string())
print(f"RasterPatchDataset smoke test passed for all three patch sizes: {smoke_test_passed}")